In [ ]:
import pandas as pd
import numpy as np

In [ ]:
df = pd.read_csv('/content/drive/MyDrive/buoc5_english_only/btc_tweet_english_buoc5.csv')

/tmp/ipykernel_8191/2812839809.py:1: DtypeWarning: Columns (4) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('/content/drive/MyDrive/buoc5_english_only/btc_tweet_english_buoc5.csv')


In [ ]:
df.shape

(3205624, 6)

In [ ]:
df.head()

,user_name,user_followers,date,text,is_retweet,text_clean
0,DeSota Wilson,8534.0,2021-02-10 23:59:04+00:00,Blue Ridge Bank shares halted by NYSE after #b...,False,Blue Ridge Bank shares halted by NYSE after #b...
1,CryptoND,6769.0,2021-02-10 23:58:48+00:00,"😎 Today, that's this #Thursday, we will do a ""...",False,"smiling face with sunglasses Today, that's thi..."
2,Tdlmatias,128.0,2021-02-10 23:54:48+00:00,"Guys evening, I have read this article about B...",False,"Guys evening, I have read this article about B..."
3,Crypto is the future,625.0,2021-02-10 23:54:33+00:00,$BTC A big chance in a billion! Price: \487264...,False,$BTC A big chance in a billion! Price: \487264...
4,Alex Kirchmaier 🇦🇹🇸🇪 #FactsSuperspreader,1249.0,2021-02-10 23:54:06+00:00,This network is secured by 9 508 nodes as of t...,False,This network is secured by 9 508 nodes as of t...


6 - Chuẩn hóa văn bản cuối cùng

In [ ]:
import re

# =====================================================================
# COMPILE REGEX CHO KÝ TỰ ĐƯỢC PHÉP GIỮ LẠI (TỐI ƯU HIỆU NĂNG)
# =====================================================================
# Chỉ giữ lại: chữ cái Latin (a-zA-Z), khoảng trắng (\s), các dấu câu hỗ trợ biểu cảm của VADER (! ? . , '),
# và ký tự '#' để bảo toàn cấu trúc các market hashtag cốt lõi (như #BTC, #Bitcoin) đã phân loại ở Bước 3.
ALLOWED_CHARS_REGEX = re.compile(r"[^a-zA-Z\s!\?.,'#]")


def normalize_text(text: str) -> str:
    """
    Bước 6 — Chuẩn hóa văn bản cuối cùng (Final Text Normalization).
    (Giữ nguyên viết hoa/viết thường và Stopwords để phục vụ tối đa cho VADER & BERTweet)
    """
    if not isinstance(text, str) or not text.strip():
        return ''

    # 1. Loại bỏ số và ký tự lạ (Giữ lại chữ cái, khoảng trắng, !, ?, ., ,, ' và #)
    text = ALLOWED_CHARS_REGEX.sub('', text)

    # 2. Chuẩn hóa các khoảng trắng thừa sinh ra trong quá trình lọc ký tự
    text = re.sub(r'\s+', ' ', text).strip()

    return text


def apply_normalization(df: pd.DataFrame, report: bool = True) -> pd.DataFrame:
    """
    Áp dụng chuẩn hóa văn bản cuối cùng lên cột 'text_clean'.
    """
    df = df.copy()
    initial_count = len(df)

    # Áp dụng hàm chuẩn hóa
    df['text_clean'] = df['text_clean'].apply(normalize_text)

    # TỐI ƯU: Sau khi lọc ký tự đặc biệt, có thể có một số tweet chỉ chứa ký tự lạ
    # (ví dụ: các tweet toàn icon lạ hoặc link đã bị xóa) bị biến thành chuỗi rỗng.
    # Chúng ta lọc bỏ các dòng rỗng này để dọn sạch tuyệt đối dữ liệu trước khi đưa vào mô hình.
    df = df[df['text_clean'].str.strip().str.len() > 0]

    dropped_count = initial_count - len(df)
    retention = (len(df) / initial_count) * 100 if initial_count > 0 else 0

    if report:
        print('=' * 65)
        print('    BÁO CÁO CHUẨN HÓA VĂN BẢN CUỐI CÙNG — BƯỚC 6')
        print('=' * 65)
        print(f"  Tweet đầu vào                     : {initial_count:>10,}")
        print(f"  Tweet bị rỗng sau chuẩn hóa (loại): {dropped_count:>10,}")
        print(f"  Tweet sạch hoàn toàn giữ lại      : {len(df):>10,}")
        print(f"  Tỉ lệ giữ lại của bước này        : {retention:>9.1f}%")
        print('=' * 65)
        print('\n  Ví dụ mẫu tweet sau khi chuẩn hóa cuối cùng:')
        samples = df[['text', 'text_clean']].head(3)
        for i, row in samples.iterrows():
            print(f'\n  [{i}] Gốc : {row["text"][:120]}')
            print(f'      Sạch: {row["text_clean"][:120]}')
        print('=' * 65)

    return df.reset_index(drop=True)

In [ ]:
# Thực thi Bước 6
df = apply_normalization(df)

    BÁO CÁO CHUẨN HÓA VĂN BẢN CUỐI CÙNG — BƯỚC 6
  Tweet đầu vào                     :  3,205,624
  Tweet bị rỗng sau chuẩn hóa (loại):          8
  Tweet sạch hoàn toàn giữ lại      :  3,205,616
  Tỉ lệ giữ lại của bước này        :     100.0%

  Ví dụ mẫu tweet sau khi chuẩn hóa cuối cùng:

  [0] Gốc : Blue Ridge Bank shares halted by NYSE after #bitcoin ATM announcement https://t.co/xaaZmaJKiV @MyBlueRidgeBank… https://
      Sạch: Blue Ridge Bank shares halted by NYSE after #bitcoin ATM announcement

  [1] Gốc : 😎 Today, that's this #Thursday, we will do a "🎬 Take 2" with our friend @LeoWandersleb, #Btc #wallet #security expe… htt
      Sạch: smiling face with sunglasses Today, that's this Thursday , we will do a clapper board Take with our friend , #Btc wallet

  [2] Gốc : Guys evening, I have read this article about BTC and would like to share with you all - https://t.co/QxCZgmuy3B… https:/
      Sạch: Guys evening, I have read this article about BTC and would like to share with 

7 - Tính trọng số ảnh hưởng theo user_followers

In [ ]:
# Đảm bảo cột user_followers ở định dạng số, các giá trị bị lỗi/khuyết thiếu sẽ được điền 0
df['user_followers'] = pd.to_numeric(df['user_followers'], errors='coerce').fillna(0)

# Tính trọng số ảnh hưởng theo công thức: Weight = ln(followers + 1) + 1
df['weight'] = np.log(df['user_followers'] + 1) + 1

# Hiển thị thống kê nhanh để kiểm tra kết quả
print(f"Tổng số bản ghi: {len(df):,}")
print(f"Trọng số lớn nhất: {df['weight'].max():.4f}")
print(f"Trọng số nhỏ nhất: {df['weight'].min():.4f}")
print(f"Trọng số trung bình: {df['weight'].mean():.4f}")

# Xem trước vài dòng dữ liệu
df[['user_name', 'user_followers', 'weight']].head()

Tổng số bản ghi: 3,205,616
Trọng số lớn nhất: 17.9990
Trọng số nhỏ nhất: 1.0000
Trọng số trung bình: 6.7291


,user_name,user_followers,weight
0,DeSota Wilson,8534.0,10.051931
1,CryptoND,6769.0,9.820256
2,Tdlmatias,128.0,5.859812
3,Crypto is the future,625.0,7.439350
4,Alex Kirchmaier 🇦🇹🇸🇪 #FactsSuperspreader,1249.0,8.130899


In [ ]:
df.head(10)

,user_name,user_followers,date,text,is_retweet,text_clean,weight
0,DeSota Wilson,8534.0,2021-02-10 23:59:04+00:00,Blue Ridge Bank shares halted by NYSE after #b...,False,Blue Ridge Bank shares halted by NYSE after #b...,10.051931
1,CryptoND,6769.0,2021-02-10 23:58:48+00:00,"😎 Today, that's this #Thursday, we will do a ""...",False,"smiling face with sunglasses Today, that's thi...",9.820256
2,Tdlmatias,128.0,2021-02-10 23:54:48+00:00,"Guys evening, I have read this article about B...",False,"Guys evening, I have read this article about B...",5.859812
3,Crypto is the future,625.0,2021-02-10 23:54:33+00:00,$BTC A big chance in a billion! Price: \487264...,False,BTC A big chance in a billion! Price . #Bitcoi...,7.439350
4,Alex Kirchmaier 🇦🇹🇸🇪 #FactsSuperspreader,1249.0,2021-02-10 23:54:06+00:00,This network is secured by 9 508 nodes as of t...,False,This network is secured by nodes as of today. ...,8.130899
5,ZerrBenz™ ⚔ ✪ 20732,742.0,2021-02-10 23:53:30+00:00,💹 Trade #Crypto on #Binance \n\n📌 Enjoy #Cashb...,False,chart increasing with yen Trade #Crypto on Bin...,7.610696
6,Mikcoin,104.0,2021-02-10 23:52:25+00:00,#BTC #Bitcoin #Ethereum #ETH #Crypto #cryptotr...,False,#BTC #Bitcoin #Ethereum #ETH #Crypto cryptotra...,5.653960
7,DeSota Wilson,8534.0,2021-02-10 23:52:08+00:00,.@Tesla’s #bitcoin investment is revolutionary...,False,.'s #bitcoin investment is revolutionary for #...,10.051931
8,@massumeh18 #RefinedWarrior #Activist,1159.0,2021-02-10 23:52:04+00:00,Annnd #btc #Bitcoin is headed even higher now....,False,Annnd #btc #Bitcoin is headed even higher now...,8.056175
9,CPUcoin,5097.0,2021-02-10 23:50:59+00:00,Join our first virtual crypto meetup of 2021 -...,False,Join our first virtual crypto meetup of Crypto...,9.536604


8 - Tính điểm cảm xúc

Phân tích cảm xúc bằng VADER

In [ ]:
!pip install vaderSentiment tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.0/126.0 kB 1.2 MB/s eta 0:00:00


In [ ]:
# =====================================================================
# BƯỚC 8 — PHÂN TÍCH CẢM XÚC BẰNG VADER
# =====================================================================

from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from tqdm import tqdm

# 1. Khởi tạo công cụ phân tích cảm xúc VADER
analyzer = SentimentIntensityAnalyzer()

# 2. Sử dụng List Comprehension kết hợp với tqdm để tối ưu tốc độ xử lý
# Vì tập dữ liệu có tới hơn 3.2 triệu dòng, dùng list comprehension sẽ nhanh hơn từ 3x - 5x
# so với dùng df['text_clean'].apply() truyền thống, đồng thời tqdm sẽ hiển thị thanh tiến trình trực quan.
print("Đang bắt đầu phân tích điểm cảm xúc bằng VADER cho ~3.2 triệu tweet...")

df['vader_compound'] = [
    analyzer.polarity_scores(text)['compound']
    for text in tqdm(df['text_clean'], desc="Đang chạy VADER")
]

# 3. Báo cáo thống kê nhanh điểm cảm xúc thu được
print("\n" + "="*65)
print("    BÁO CÁO KẾT QUẢ PHÂN TÍCH CẢM XÚC VADER — BƯỚC 8")
print("="*65)
print(df['vader_compound'].describe())
print("="*65)
# Thống kê phân loại theo ngưỡng tiêu chuẩn của VADER (+/- 0.05)
pos_ratio = len(df[df['vader_compound'] >= 0.05]) / len(df) * 100
neg_ratio = len(df[df['vader_compound'] <= -0.05]) / len(df) * 100
neu_ratio = len(df[(df['vader_compound'] > -0.05) & (df['vader_compound'] < 0.05)]) / len(df) * 100

print(f"Tỉ lệ tweet tích cực (compound >= 0.05)  : {pos_ratio:.1f}%")
print(f"Tỉ lệ tweet tiêu cực (compound <= -0.05) : {neg_ratio:.1f}%")
print(f"Tỉ lệ tweet trung lập (-0.05 < c < 0.05) : {neu_ratio:.1f}%")
print("="*65)

# 4. Xem trước kết quả mẫu
df[['text_clean', 'weight', 'vader_compound']].head()

Đang bắt đầu phân tích điểm cảm xúc bằng VADER cho ~3.2 triệu tweet...


Đang chạy VADER: 100%|██████████| 3205616/3205616 [07:29<00:00, 7133.70it/s]



    BÁO CÁO KẾT QUẢ PHÂN TÍCH CẢM XÚC VADER — BƯỚC 8
count    3.205616e+06
mean     2.264218e-01
std      4.529217e-01
min     -9.998000e-01
25%      0.000000e+00
50%      1.779000e-01
75%      6.124000e-01
max      1.000000e+00
Name: vader_compound, dtype: float64
Tỉ lệ tweet tích cực (compound >= 0.05)  : 52.6%
Tỉ lệ tweet tiêu cực (compound <= -0.05) : 18.1%
Tỉ lệ tweet trung lập (-0.05 < c < 0.05) : 29.3%


,text_clean,weight,vader_compound
0,Blue Ridge Bank shares halted by NYSE after #b...,10.051931,0.2960
1,"smiling face with sunglasses Today, that's thi...",9.820256,0.8225
2,"Guys evening, I have read this article about B...",5.859812,0.5719
3,BTC A big chance in a billion! Price . #Bitcoi...,7.439350,0.3164
4,This network is secured by nodes as of today. ...,8.130899,-0.2023


In [ ]:
df.to_csv('/content/drive/MyDrive/buoc8_sentiment_analysis/btc_tweet_VADER.csv')

Phân tích cảm xúc bằng BERTweet Sentiment Analysis

In [ ]:
!pip install pysentimiento transformers torch tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 37.1 MB/s eta 0:00:00


In [ ]:
import os
import gc
import torch
import torch.nn.functional as F
from torch.cuda.amp import autocast
import pandas as pd
import numpy as np
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from tqdm import tqdm

In [ ]:
df = pd.read_csv('/content/drive/MyDrive/buoc8_sentiment_analysis/btc_tweet_VADER.csv')

df.head()

/tmp/ipykernel_828/3435585515.py:1: DtypeWarning: Columns (5) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('/content/drive/MyDrive/buoc8_sentiment_analysis/btc_tweet_VADER.csv')


,Unnamed: 0,user_name,user_followers,date,text,is_retweet,text_clean,weight,vader_compound
0,0,DeSota Wilson,8534.0,2021-02-10 23:59:04+00:00,Blue Ridge Bank shares halted by NYSE after #b...,False,Blue Ridge Bank shares halted by NYSE after #b...,10.051931,0.2960
1,1,CryptoND,6769.0,2021-02-10 23:58:48+00:00,"😎 Today, that's this #Thursday, we will do a ""...",False,"smiling face with sunglasses Today, that's thi...",9.820256,0.8225
2,2,Tdlmatias,128.0,2021-02-10 23:54:48+00:00,"Guys evening, I have read this article about B...",False,"Guys evening, I have read this article about B...",5.859812,0.5719
3,3,Crypto is the future,625.0,2021-02-10 23:54:33+00:00,$BTC A big chance in a billion! Price: \487264...,False,BTC A big chance in a billion! Price . #Bitcoi...,7.439350,0.3164
4,4,Alex Kirchmaier 🇦🇹🇸🇪 #FactsSuperspreader,1249.0,2021-02-10 23:54:06+00:00,This network is secured by 9 508 nodes as of t...,False,This network is secured by nodes as of today. ...,8.130899,-0.2023


In [ ]:
# ══════════════════════════════════════════════════════════════════════
# BƯỚC 8 — BERTWEET SA | FULL GPU T4 OPTIMIZATION
# Kỹ thuật: FP16 autocast · cuDNN benchmark · Dynamic+Sorted Padding
#           non_blocking transfer · no_grad · Unique · Checkpoint Resume
# ══════════════════════════════════════════════════════════════════════

# ─── CẤU HÌNH ────────────────────────────────────────────────────────
CHECKPOINT_PATH = "/content/drive/MyDrive/buoc8_sentiment_analysis/bertweet_sa_unique_checkpoint.csv"
MODEL_NAME      = "finiteautomata/bertweet-base-sentiment-analysis"
BATCH_SIZE      = 512   # FP16 giải phóng VRAM → batch gấp đôi
SAVE_EVERY      = 20    # Lưu Drive sau mỗi 20 batches (~10 240 tweet)

# ─── 1. THIẾT LẬP CUDA TỐI ƯU ────────────────────────────────────────
device = "cuda" if torch.cuda.is_available() else "cpu"
if device == "cuda":
    # Cho phép cuDNN tự lựa chọn thuật toán convolution nhanh nhất
    # phù hợp với kích thước input thực tế của batch
    torch.backends.cudnn.benchmark = True
    gpu_name = torch.cuda.get_device_name(0)
    gpu_vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"✅ GPU : {gpu_name}")
    print(f"✅ VRAM: {gpu_vram:.1f} GB")
else:
    print("⚠️  Không tìm thấy GPU! Hãy chuyển Runtime sang GPU T4 trước.")

# ─── 2. LỌC UNIQUE TWEETS ────────────────────────────────────────────
print("\n[1/5] Lọc tweet trùng lặp...")
df['text_clean'] = df['text_clean'].fillna('').astype(str)
df_unique = df[['text_clean']].drop_duplicates().reset_index(drop=True)
print(f"      {len(df):,} dòng gốc  →  {len(df_unique):,} dòng độc nhất "
      f"(tiết kiệm {len(df)-len(df_unique):,} lần tính toán)")

# ─── 3. KHÔI PHỤC CHECKPOINT ─────────────────────────────────────────
print("\n[2/5] Kiểm tra checkpoint...")
done_texts = set()
if os.path.exists(CHECKPOINT_PATH):
    try:
        df_ckpt = pd.read_csv(CHECKPOINT_PATH)
        df_ckpt['text_clean'] = df_ckpt['text_clean'].fillna('').astype(str)
        done_texts = set(df_ckpt['text_clean'])
        print(f"      Phục hồi được {len(done_texts):,} dòng đã xử lý trước đó.")
    except Exception as e:
        print(f"      Lỗi đọc checkpoint ({e}), sẽ tạo mới.")

df_todo = df_unique[~df_unique['text_clean'].isin(done_texts)].reset_index(drop=True)
print(f"      Còn lại cần chạy: {len(df_todo):,} dòng.")

# ─── 4. TẢI MÔ HÌNH ──────────────────────────────────────────────────
if len(df_todo) > 0:
    print("\n[3/5] Tải mô hình BERTweet...")
    # use_fast=True: dùng Rust tokenizer (nhanh hơn ~10x so với Python tokenizer)
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)
    model.eval().to(device)

    labels_list = [model.config.id2label[i] for i in range(3)]
    pos_idx = labels_list.index('POS')
    neg_idx = labels_list.index('NEG')
    neu_idx = labels_list.index('NEU')
    print(f"      Nhãn: {labels_list}  |  pos={pos_idx} neg={neg_idx} neu={neu_idx}")

    # ─── 5. SORT BY LENGTH → GIẢM PADDING WASTE ─────────────────────
    # Sắp xếp tweet theo chiều dài tăng dần trước khi tạo batch.
    # Mỗi batch sẽ chứa các tweet có độ dài gần nhau → padding overhead
    # giảm tối đa, GPU không phải xử lý hàng chục token [PAD] vô nghĩa.
    print("\n[4/5] Sắp xếp tweet theo độ dài để tối ưu Dynamic Padding...")
    texts_sorted = sorted(df_todo['text_clean'].tolist(), key=len)
    print(f"      Độ dài ngắn nhất: {len(texts_sorted[0])} ký tự  |  "
          f"Dài nhất: {len(texts_sorted[-1])} ký tự")

    # ─── 6. CHẠY DỰ ĐOÁN ─────────────────────────────────────────────
    print(f"\n[5/5] Chạy BERTweet (FP16 autocast | sorted batch {BATCH_SIZE})...")
    temp_results = []

    with torch.no_grad():
        for i in tqdm(range(0, len(texts_sorted), BATCH_SIZE), desc="BERTweet GPU"):
            batch = texts_sorted[i : i + BATCH_SIZE]

            # Tokenize: padding về tweet DÀI NHẤT TRONG BATCH (Dynamic Padding)
            # non_blocking=True: CPU→GPU transfer bất đồng bộ, không chặn luồng
            inputs = tokenizer(
                batch,
                padding=True,
                truncation=True,
                max_length=128,
                return_tensors="pt"
            ).to(device, non_blocking=True)

            # autocast: tự động dùng FP16 ở các phép tính phù hợp (matrix mul,
            # attention), giữ FP32 ở các phép tính nhạy cảm (softmax, norm).
            # Đây là cách chuyên nghiệp nhất để bật FP16 trên T4 Tensor Cores.
            with autocast():
                logits = model(**inputs).logits

            # Softmax ở FP32 để tránh mất độ chính xác số học
            probs = F.softmax(logits.float(), dim=-1).cpu().numpy()

            for txt, prob in zip(batch, probs):
                temp_results.append({
                    'text_clean'   : txt,
                    'bert_pos'     : float(prob[pos_idx]),
                    'bert_neg'     : float(prob[neg_idx]),
                    'bert_neu'     : float(prob[neu_idx]),
                    'bert_compound': float(prob[pos_idx] - prob[neg_idx])
                })

            # Lưu checkpoint định kỳ
            batch_idx = (i // BATCH_SIZE) + 1
            is_last   = (i + BATCH_SIZE) >= len(texts_sorted)
            if batch_idx % SAVE_EVERY == 0 or is_last:
                df_temp    = pd.DataFrame(temp_results)
                file_exists = os.path.exists(CHECKPOINT_PATH)
                df_temp.to_csv(CHECKPOINT_PATH, mode='a',
                               header=not file_exists, index=False)
                temp_results.clear()

                # Dọn VRAM định kỳ để tránh memory leak
                gc.collect()
                torch.cuda.empty_cache()

    print(f"\n✅ Hoàn thành! Checkpoint lưu tại:\n   {CHECKPOINT_PATH}")

else:
    print("\n✅ Toàn bộ dòng đã xử lý. Bỏ qua bước chạy BERT.")

# ─── 7. ÁNH XẠ NGƯỢC VỀ DATAFRAME GỐC ───────────────────────────────
print("\nÁnh xạ kết quả về 3.2 triệu dòng gốc...")
df_res = pd.read_csv(CHECKPOINT_PATH)
df_res['text_clean'] = df_res['text_clean'].fillna('').astype(str)

for col in ['bert_pos', 'bert_neg', 'bert_neu', 'bert_compound']:
    df[col] = df['text_clean'].map(dict(zip(df_res['text_clean'], df_res[col])))

print("\n" + "═"*60)
print("  BÁO CÁO BERTWEET SA — HOÀN THÀNH")
print("═"*60)
print(df['bert_compound'].describe().round(4))
print("═"*60)
df[['text_clean', 'weight', 'vader_compound', 'bert_compound']].head()

⚠️  Không tìm thấy GPU! Hãy chuyển Runtime sang GPU T4 trước.

[1/5] Lọc tweet trùng lặp...
      3,205,616 dòng gốc  →  2,855,464 dòng độc nhất (tiết kiệm 350,152 lần tính toán)

[2/5] Kiểm tra checkpoint...
      Phục hồi được 2,855,464 dòng đã xử lý trước đó.
      Còn lại cần chạy: 0 dòng.

✅ Toàn bộ dòng đã xử lý. Bỏ qua bước chạy BERT.

Ánh xạ kết quả về 3.2 triệu dòng gốc...

════════════════════════════════════════════════════════════
  BÁO CÁO BERTWEET SA — HOÀN THÀNH
════════════════════════════════════════════════════════════
count    3.205616e+06
mean     2.866000e-01
std      5.385000e-01
min     -9.807000e-01
25%      2.400000e-02
50%      1.888000e-01
75%      8.366000e-01
max      9.915000e-01
Name: bert_compound, dtype: float64
════════════════════════════════════════════════════════════


,text_clean,weight,vader_compound,bert_compound
0,Blue Ridge Bank shares halted by NYSE after #b...,10.051931,0.2960,-0.698654
1,"smiling face with sunglasses Today, that's thi...",9.820256,0.8225,0.809963
2,"Guys evening, I have read this article about B...",5.859812,0.5719,0.423179
3,BTC A big chance in a billion! Price . #Bitcoi...,7.439350,0.3164,0.968326
4,This network is secured by nodes as of today. ...,8.130899,-0.2023,0.284730


In [ ]:
df.head(10)

,Unnamed: 0,user_name,user_followers,date,text,is_retweet,text_clean,weight,vader_compound,bert_pos,bert_neg,bert_neu,bert_compound
0,0,DeSota Wilson,8534.0,2021-02-10 23:59:04+00:00,Blue Ridge Bank shares halted by NYSE after #b...,False,Blue Ridge Bank shares halted by NYSE after #b...,10.051931,0.2960,0.003876,0.702530,0.293595,-0.698654
1,1,CryptoND,6769.0,2021-02-10 23:58:48+00:00,"😎 Today, that's this #Thursday, we will do a ""...",False,"smiling face with sunglasses Today, that's thi...",9.820256,0.8225,0.811350,0.001387,0.187264,0.809963
2,2,Tdlmatias,128.0,2021-02-10 23:54:48+00:00,"Guys evening, I have read this article about B...",False,"Guys evening, I have read this article about B...",5.859812,0.5719,0.425451,0.002273,0.572276,0.423179
3,3,Crypto is the future,625.0,2021-02-10 23:54:33+00:00,$BTC A big chance in a billion! Price: \487264...,False,BTC A big chance in a billion! Price . #Bitcoi...,7.439350,0.3164,0.969479,0.001154,0.029367,0.968326
4,4,Alex Kirchmaier 🇦🇹🇸🇪 #FactsSuperspreader,1249.0,2021-02-10 23:54:06+00:00,This network is secured by 9 508 nodes as of t...,False,This network is secured by nodes as of today. ...,8.130899,-0.2023,0.296775,0.012045,0.691181,0.284730
5,5,ZerrBenz™ ⚔ ✪ 20732,742.0,2021-02-10 23:53:30+00:00,💹 Trade #Crypto on #Binance \n\n📌 Enjoy #Cashb...,False,chart increasing with yen Trade #Crypto on Bin...,7.610696,0.4939,0.611585,0.001946,0.386469,0.609638
6,6,Mikcoin,104.0,2021-02-10 23:52:25+00:00,#BTC #Bitcoin #Ethereum #ETH #Crypto #cryptotr...,False,#BTC #Bitcoin #Ethereum #ETH #Crypto cryptotra...,5.653960,0.0000,0.055815,0.004792,0.939393,0.051022
7,7,DeSota Wilson,8534.0,2021-02-10 23:52:08+00:00,.@Tesla’s #bitcoin investment is revolutionary...,False,.'s #bitcoin investment is revolutionary for #...,10.051931,0.0000,0.473875,0.016490,0.509635,0.457385
8,8,@massumeh18 #RefinedWarrior #Activist,1159.0,2021-02-10 23:52:04+00:00,Annnd #btc #Bitcoin is headed even higher now....,False,Annnd #btc #Bitcoin is headed even higher now...,8.056175,0.0000,0.933062,0.001382,0.065556,0.931680
9,9,CPUcoin,5097.0,2021-02-10 23:50:59+00:00,Join our first virtual crypto meetup of 2021 -...,False,Join our first virtual crypto meetup of Crypto...,9.536604,0.3595,0.900633,0.001126,0.098241,0.899506


In [ ]:
df.to_csv('/content/drive/MyDrive/buoc8_sentiment_analysis/btc_tweet_SA.csv')

Kiểm tra dữ liệu thời gian

In [ ]:
df['date'] = pd.to_datetime(df['date'])

print(f"Từ: {df['date'].min()}")
print(f"Đến: {df['date'].max()}")
print(f"Tổng khoảng thời gian: {df['date'].max() - df['date'].min()}")

Từ: 2021-02-05 10:52:04+00:00
Đến: 2023-01-09 23:59:54+00:00
Tổng khoảng thời gian: 703 days 13:07:50


#BERTweet ED

In [ ]:
df = pd.read_csv('/content/drive/MyDrive/buoc8_sentiment_analysis/btc_tweet_SA.csv')

/tmp/ipykernel_1340/657201066.py:1: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('/content/drive/MyDrive/buoc8_sentiment_analysis/btc_tweet_SA.csv')


In [ ]:
df.head()

,Unnamed: 0.1,Unnamed: 0,user_name,user_followers,date,text,is_retweet,text_clean,weight,vader_compound,bert_pos,bert_neg,bert_neu,bert_compound
0,0,0,DeSota Wilson,8534.0,2021-02-10 23:59:04+00:00,Blue Ridge Bank shares halted by NYSE after #b...,False,Blue Ridge Bank shares halted by NYSE after #b...,10.051931,0.2960,0.003876,0.702530,0.293595,-0.698654
1,1,1,CryptoND,6769.0,2021-02-10 23:58:48+00:00,"😎 Today, that's this #Thursday, we will do a ""...",False,"smiling face with sunglasses Today, that's thi...",9.820256,0.8225,0.811350,0.001387,0.187264,0.809963
2,2,2,Tdlmatias,128.0,2021-02-10 23:54:48+00:00,"Guys evening, I have read this article about B...",False,"Guys evening, I have read this article about B...",5.859812,0.5719,0.425451,0.002273,0.572276,0.423179
3,3,3,Crypto is the future,625.0,2021-02-10 23:54:33+00:00,$BTC A big chance in a billion! Price: \487264...,False,BTC A big chance in a billion! Price . #Bitcoi...,7.439350,0.3164,0.969479,0.001154,0.029367,0.968326
4,4,4,Alex Kirchmaier 🇦🇹🇸🇪 #FactsSuperspreader,1249.0,2021-02-10 23:54:06+00:00,This network is secured by 9 508 nodes as of t...,False,This network is secured by nodes as of today. ...,8.130899,-0.2023,0.296775,0.012045,0.691181,0.284730


In [ ]:
# ══════════════════════════════════════════════════════════════════════
# BƯỚC 8 (BỔ SUNG) — BERTWEET EMOTION DETECTION (ED)
# 7 cảm xúc: joy · sadness · anger · surprise · disgust · fear · others
# Kỹ thuật: FP16 autocast · cudnn.benchmark · Sorted Padding · Checkpoint
# ══════════════════════════════════════════════════════════════════════

# ─── CẤU HÌNH (3 điểm khác so với BERTweet SA) ───────────────────────
CHECKPOINT_PATH = "/content/drive/MyDrive/buoc8_sentiment_analysis/bertweet_ed_unique_checkpoint.csv"  # [KHÁC 1]
MODEL_NAME      = "finiteautomata/bertweet-base-emotion-analysis"                                       # [KHÁC 2]
BATCH_SIZE      = 512
SAVE_EVERY      = 20

# ─── 1. THIẾT LẬP CUDA ───────────────────────────────────────────────
device = "cuda" if torch.cuda.is_available() else "cpu"
if device == "cuda":
    torch.backends.cudnn.benchmark = True
    print(f"✅ GPU : {torch.cuda.get_device_name(0)}")
    print(f"✅ VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
else:
    print("⚠️  CPU mode — sẽ chậm hơn GPU nhưng vẫn chạy được.")

# ─── 2. LỌC UNIQUE TWEETS ────────────────────────────────────────────
print("\n[1/5] Lọc tweet trùng lặp...")
df['text_clean'] = df['text_clean'].fillna('').astype(str)
df_unique = df[['text_clean']].drop_duplicates().reset_index(drop=True)
print(f"      {len(df):,} dòng gốc  →  {len(df_unique):,} dòng độc nhất")

# ─── 3. KHÔI PHỤC CHECKPOINT ─────────────────────────────────────────
print("\n[2/5] Kiểm tra checkpoint...")
done_texts = set()
if os.path.exists(CHECKPOINT_PATH):
    try:
        df_ckpt = pd.read_csv(CHECKPOINT_PATH)
        df_ckpt['text_clean'] = df_ckpt['text_clean'].fillna('').astype(str)
        done_texts = set(df_ckpt['text_clean'])
        print(f"      Phục hồi được {len(done_texts):,} dòng đã xử lý trước đó.")
    except Exception as e:
        print(f"      Lỗi đọc checkpoint ({e}), tạo mới.")

df_todo = df_unique[~df_unique['text_clean'].isin(done_texts)].reset_index(drop=True)
print(f"      Còn lại cần chạy: {len(df_todo):,} dòng.")

# ─── 4. TẢI MÔ HÌNH ──────────────────────────────────────────────────
if len(df_todo) > 0:
    print("\n[3/5] Tải mô hình BERTweet ED...")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)
    if device == "cuda":
        model = model.half()
    model = model.to(device).eval()

    # Đọc danh sách 7 nhãn cảm xúc tự động từ cấu hình model
    num_labels  = model.config.num_labels
    labels_list = [model.config.id2label[i].lower() for i in range(num_labels)]
    print(f"      Số nhãn cảm xúc: {num_labels}")
    print(f"      Danh sách nhãn : {labels_list}")

    # ─── 5. SORT BY LENGTH ───────────────────────────────────────────
    print("\n[4/5] Sắp xếp theo độ dài để tối ưu Dynamic Padding...")
    texts_sorted = sorted(df_todo['text_clean'].tolist(), key=len)

    # ─── 6. CHẠY DỰ ĐOÁN ─────────────────────────────────────────────
    print(f"\n[5/5] Chạy BERTweet ED (FP16 | Batch {BATCH_SIZE})...")
    temp_results = []

    with torch.no_grad():
        for i in tqdm(range(0, len(texts_sorted), BATCH_SIZE), desc="BERTweet ED"):
            batch = texts_sorted[i : i + BATCH_SIZE]

            inputs = tokenizer(
                batch,
                padding=True,
                truncation=True,
                max_length=128,
                return_tensors="pt"
            ).to(device, non_blocking=True)

            with autocast():
                logits = model(**inputs).logits

            probs = F.softmax(logits.float(), dim=-1).cpu().numpy()

            # [KHÁC 3]: Lưu 7 cột cảm xúc thay vì 3 cột POS/NEG/NEU
            for txt, prob in zip(batch, probs):
                row = {'text_clean': txt}
                for idx, label in enumerate(labels_list):
                    row[f'ed_{label}'] = float(prob[idx])  # ed_joy, ed_sadness, ed_anger...
                temp_results.append(row)

            # Lưu checkpoint định kỳ
            batch_idx = (i // BATCH_SIZE) + 1
            is_last   = (i + BATCH_SIZE) >= len(texts_sorted)
            if batch_idx % SAVE_EVERY == 0 or is_last:
                df_temp = pd.DataFrame(temp_results)
                file_exists = os.path.exists(CHECKPOINT_PATH)
                df_temp.to_csv(CHECKPOINT_PATH, mode='a',
                               header=not file_exists, index=False)
                temp_results.clear()
                gc.collect()
                if device == "cuda":
                    torch.cuda.empty_cache()

    print(f"\n✅ Hoàn thành! Checkpoint: {CHECKPOINT_PATH}")
else:
    print("\n✅ Toàn bộ dòng đã xử lý. Bỏ qua bước chạy.")

# ─── 7. ÁNH XẠ NGƯỢC VỀ DATAFRAME GỐC ───────────────────────────────
print("\nÁnh xạ kết quả về 3.2 triệu dòng gốc...")
df_res = pd.read_csv(CHECKPOINT_PATH)
df_res['text_clean'] = df_res['text_clean'].fillna('').astype(str)

ed_cols = [c for c in df_res.columns if c.startswith('ed_')]
for col in ed_cols:
    df[col] = df['text_clean'].map(dict(zip(df_res['text_clean'], df_res[col])))

print("\n" + "═"*60)
print("  BÁO CÁO BERTWEET ED — HOÀN THÀNH")
print("═"*60)
print(f"  Các cột cảm xúc đã thêm: {ed_cols}")
print(df[ed_cols].describe().round(4))
print("═"*60)
df[['text_clean'] + ed_cols].head()

✅ GPU : Tesla T4
✅ VRAM: 14.6 GB

[1/5] Lọc tweet trùng lặp...
      3,205,616 dòng gốc  →  2,855,464 dòng độc nhất

[2/5] Kiểm tra checkpoint...
      Còn lại cần chạy: 2,855,464 dòng.

[3/5] Tải mô hình BERTweet ED...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/999 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/295 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

...

RobertaForSequenceClassification LOAD REPORT from: finiteautomata/bertweet-base-emotion-analysis
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


      Số nhãn cảm xúc: 7
      Danh sách nhãn : ['others', 'joy', 'sadness', 'anger', 'surprise', 'disgust', 'fear']

[4/5] Sắp xếp theo độ dài để tối ưu Dynamic Padding...

[5/5] Chạy BERTweet ED (FP16 | Batch 512)...


BERTweet ED:   0%|          | 0/5578 [00:00<?, ?it/s]/tmp/ipykernel_1340/2966711457.py:78: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
BERTweet ED: 100%|██████████| 5578/5578 [1:13:06<00:00,  1.27it/s]



✅ Hoàn thành! Checkpoint: /content/drive/MyDrive/buoc8_sentiment_analysis/bertweet_ed_unique_checkpoint.csv

Ánh xạ kết quả về 3.2 triệu dòng gốc...

════════════════════════════════════════════════════════════
  BÁO CÁO BERTWEET ED — HOÀN THÀNH
════════════════════════════════════════════════════════════
  Các cột cảm xúc đã thêm: ['ed_others', 'ed_joy', 'ed_sadness', 'ed_anger', 'ed_surprise', 'ed_disgust', 'ed_fear']
          ed_others        ed_joy    ed_sadness      ed_anger   ed_surprise  \
count  3.205616e+06  3.205616e+06  3.205616e+06  3.205616e+06  3.205616e+06   
mean   7.643000e-01  1.491000e-01  4.700000e-03  1.290000e-02  8.500000e-03   
std    3.428000e-01  2.929000e-01  4.970000e-02  9.150000e-02  5.840000e-02   
min    1.000000e-03  6.000000e-04  4.000000e-04  6.000000e-04  6.000000e-04   
25%    6.976000e-01  9.200000e-03  1.100000e-03  1.400000e-03  2.400000e-03   
50%    9.621000e-01  1.580000e-02  1.300000e-03  1.700000e-03  3.000000e-03   
75%    9.751000e-01  5

,text_clean,ed_others,ed_joy,ed_sadness,ed_anger,ed_surprise,ed_disgust,ed_fear
0,Blue Ridge Bank shares halted by NYSE after #b...,0.942648,0.021046,0.007805,0.001809,0.003886,0.003447,0.019360
1,"smiling face with sunglasses Today, that's thi...",0.577389,0.411033,0.001134,0.002490,0.003445,0.001673,0.002835
2,"Guys evening, I have read this article about B...",0.705347,0.270875,0.001586,0.001823,0.016833,0.001167,0.002369
3,BTC A big chance in a billion! Price . #Bitcoi...,0.974448,0.012805,0.000759,0.001579,0.005602,0.002624,0.002182
4,This network is secured by nodes as of today. ...,0.969344,0.018899,0.001749,0.000984,0.002400,0.001908,0.004715


In [ ]:
df.head()

,Unnamed: 0.1,Unnamed: 0,user_name,user_followers,date,text,is_retweet,text_clean,weight,vader_compound,...,bert_neg,bert_neu,bert_compound,ed_others,ed_joy,ed_sadness,ed_anger,ed_surprise,ed_disgust,ed_fear
0,0,0,DeSota Wilson,8534.0,2021-02-10 23:59:04+00:00,Blue Ridge Bank shares halted by NYSE after #b...,False,Blue Ridge Bank shares halted by NYSE after #b...,10.051931,0.2960,...,0.702530,0.293595,-0.698654,0.942648,0.021046,0.007805,0.001809,0.003886,0.003447,0.019360
1,1,1,CryptoND,6769.0,2021-02-10 23:58:48+00:00,"😎 Today, that's this #Thursday, we will do a ""...",False,"smiling face with sunglasses Today, that's thi...",9.820256,0.8225,...,0.001387,0.187264,0.809963,0.577389,0.411033,0.001134,0.002490,0.003445,0.001673,0.002835
2,2,2,Tdlmatias,128.0,2021-02-10 23:54:48+00:00,"Guys evening, I have read this article about B...",False,"Guys evening, I have read this article about B...",5.859812,0.5719,...,0.002273,0.572276,0.423179,0.705347,0.270875,0.001586,0.001823,0.016833,0.001167,0.002369
3,3,3,Crypto is the future,625.0,2021-02-10 23:54:33+00:00,$BTC A big chance in a billion! Price: \487264...,False,BTC A big chance in a billion! Price . #Bitcoi...,7.439350,0.3164,...,0.001154,0.029367,0.968326,0.974448,0.012805,0.000759,0.001579,0.005602,0.002624,0.002182
4,4,4,Alex Kirchmaier 🇦🇹🇸🇪 #FactsSuperspreader,1249.0,2021-02-10 23:54:06+00:00,This network is secured by 9 508 nodes as of t...,False,This network is secured by nodes as of today. ...,8.130899,-0.2023,...,0.012045,0.691181,0.284730,0.969344,0.018899,0.001749,0.000984,0.002400,0.001908,0.004715


In [ ]:
df.drop(columns=['Unnamed: 0.1', 'Unnamed: 0'], inplace=True)

In [ ]:
df.head()

,user_name,user_followers,date,text,is_retweet,text_clean,weight,vader_compound,bert_pos,bert_neg,bert_neu,bert_compound,ed_others,ed_joy,ed_sadness,ed_anger,ed_surprise,ed_disgust,ed_fear
0,DeSota Wilson,8534.0,2021-02-10 23:59:04+00:00,Blue Ridge Bank shares halted by NYSE after #b...,False,Blue Ridge Bank shares halted by NYSE after #b...,10.051931,0.2960,0.003876,0.702530,0.293595,-0.698654,0.942648,0.021046,0.007805,0.001809,0.003886,0.003447,0.019360
1,CryptoND,6769.0,2021-02-10 23:58:48+00:00,"😎 Today, that's this #Thursday, we will do a ""...",False,"smiling face with sunglasses Today, that's thi...",9.820256,0.8225,0.811350,0.001387,0.187264,0.809963,0.577389,0.411033,0.001134,0.002490,0.003445,0.001673,0.002835
2,Tdlmatias,128.0,2021-02-10 23:54:48+00:00,"Guys evening, I have read this article about B...",False,"Guys evening, I have read this article about B...",5.859812,0.5719,0.425451,0.002273,0.572276,0.423179,0.705347,0.270875,0.001586,0.001823,0.016833,0.001167,0.002369
3,Crypto is the future,625.0,2021-02-10 23:54:33+00:00,$BTC A big chance in a billion! Price: \487264...,False,BTC A big chance in a billion! Price . #Bitcoi...,7.439350,0.3164,0.969479,0.001154,0.029367,0.968326,0.974448,0.012805,0.000759,0.001579,0.005602,0.002624,0.002182
4,Alex Kirchmaier 🇦🇹🇸🇪 #FactsSuperspreader,1249.0,2021-02-10 23:54:06+00:00,This network is secured by 9 508 nodes as of t...,False,This network is secured by nodes as of today. ...,8.130899,-0.2023,0.296775,0.012045,0.691181,0.284730,0.969344,0.018899,0.001749,0.000984,0.002400,0.001908,0.004715


In [ ]:
df.to_csv('/content/drive/MyDrive/buoc8_sentiment_analysis/btc_tweet_SA_ED.csv', index=False)